In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from scipy.interpolate import griddata
from pyproj import Transformer, CRS
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling

import reprojection

Open Dataset

In [ ]:
path = "CO/S5P_OFFL_L2__CO_____20240701T092714_20240701T110844_34802_03_020600_20240702T231245.nc.zip"
ds = xr.open_dataset(path, engine="netcdf4", group = "PRODUCT")

Get a spatial extent in lat lon over Austria

In [ ]:
transformer = Transformer.from_crs("EPSG:27704", "EPSG:4326", always_xy=True)
x,y = 5700000, 2100000
lon_max, lat_max = transformer.transform(x, y)
x,y = 4200000, 900000
lon_min, lat_min = transformer.transform(x,y)

Load the data and rearrange it to fit needs

In [ ]:
ds = ds.load()
ds = ds.set_coords(["longitude", "latitude"])
delta_time  = ds[["delta_time", "latitude", "longitude"]]

ds = ds.drop_vars(["delta_time", "time_utc"])

Filter according to a mask. All values outside of Austrias spatial extent and quality value under 0.5 will not be regarded.

In [ ]:
mask = (
    (ds["latitude"] >= lat_min) & (ds["latitude"] <= lat_max) &
    (ds["longitude"] >= lon_min) & (ds["longitude"] <= lon_max) & 
    (ds["qa_value"] >= 0.5)
)

ds_subset = ds.where(mask, drop=True) 

Add epsg code to the dataset, and remove unnecessary dimensions

In [ ]:
ds_subset = ds_subset.rio.write_crs("EPSG:4326", inplace=False)
ds_band = ds_subset.squeeze()

save original lat lon values and save max and min values

In [ ]:
lat_original = ds_band["latitude"]
lon_original = ds_band["longitude"]

In [ ]:
lat_min, lat_max = lat_original.min().item(), lat_original.max().item()
lon_min, lon_max = lon_original.min().item(), lon_original.max().item()
lat_min, lat_max, lon_min, lon_max

The original data is in swath coordinates, the next steps will resample the data to a grid with 0.1° resolution

In [ ]:
target_resolution = 0.1

n_points_lat_full = int(np.ceil((lat_max - lat_min) / target_resolution)) + 1
n_points_lon_full = int(np.ceil((lon_max - lon_min) / target_resolution)) + 1
target_lat_full = np.linspace(lat_min, lat_max, n_points_lat_full)
target_lon_full = np.linspace(lon_min, lon_max, n_points_lon_full)
lon_grid_full, lat_grid_full = np.meshgrid(target_lon_full, target_lat_full)

The gridded data is saved as a dictionary with the variable names as keys

In [ ]:
gridded_data = {}

for var in ds_band.data_vars:
    gridded = griddata((lon_original.values.flatten(), lat_original.values.flatten()),
                                    ds_band[var].values.flatten(),
                                    (lon_grid_full, lat_grid_full),
                                    method="linear")

    gridded_data[var] = gridded

Define a function to reproject the data to equi7 grid with a 10km resolution

In [ ]:
def reproject_numpy_to_equi7_eu_10km(
    data, lons, lats, nodata=np.nan, resampling="bilinear"
):

    assert data.ndim == 2 and data.shape == (len(lats), len(lons))

    if lats[0] < lats[-1]:
        data = data[::-1, :]
        lats = lats[::-1]

    xres = float(abs(lons[1] - lons[0]))
    yres = float(abs(lats[1] - lats[0]))
    west = float(lons.min())
    north = float(lats.max())
    src_transform = from_origin(west - xres/2, north + yres/2, xres, yres)
    src_crs = "EPSG:4326"

    dst_crs = "EPSG:27704"
    out_res = 10000.0

    left, bottom, right, top = rasterio.transform.array_bounds(
        data.shape[0], data.shape[1], src_transform
    )
    dst_transform, dst_w, dst_h = calculate_default_transform(
        src_crs, dst_crs, data.shape[1], data.shape[0],
        left, bottom, right, top, resolution=out_res
    )

    from math import floor
    x0 = floor(dst_transform.c / out_res) * out_res
    y0 = floor(dst_transform.f / out_res) * out_res
    dst_transform = rasterio.Affine(out_res, 0, x0, 0, -out_res, y0)

    dst = np.full((dst_h, dst_w), nodata, dtype=data.dtype)
    reproject(
        source=data,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=getattr(Resampling, resampling),
        src_nodata=nodata,
        dst_nodata=nodata,
    )
    return dst, dst_transform, dst_crs

Each data variable is then reprojected to equi7grid

In [ ]:
reprojected = {}

for var in gridded_data:
    arr_10km, tr_10km, crs_10km = reproject_numpy_to_equi7_eu_10km(
        gridded_data[var], target_lon_full, target_lat_full,
        nodata=np.nan,
        resampling="bilinear"  
    )

    reprojected[var] = (("y", "x"),arr_10km)

We need to define a function to get us the coordinate values of the reprojected data

In [ ]:
def pixel_center_coords(transform, width, height):
    """
    Returns 2D arrays X, Y with the projected coordinates
    of each pixel center given an affine transform.
    """
    # Column indices (0..width-1)
    cols = np.arange(width)
    # Row indices (0..height-1)
    rows = np.arange(height)

    # Convert pixel indices to map coords (center positions)
    xs = transform.c + cols * transform.a + transform.b * 0
    ys = transform.f + rows * transform.e + transform.d * 0

    # Meshgrid to 2D arrays
    #X, Y = np.meshgrid(xs, ys)
    return xs, ys

xs, ys = pixel_center_coords(tr_10km, arr_10km.shape[1], arr_10km.shape[0])

The sensing date is extracted

In [ ]:
sensing_time = delta_time["delta_time"].values[0][0].astype("datetime64[D]")

A dataset is created for the data variables

In [ ]:
ds_equi7 = xr.Dataset(reprojected,
                      coords={
                              "x":("x", xs),
                              "y":("y", ys),
                              })

And clipped to the spatial extent of Austria, also a time dimension is added

In [ ]:
ds_aut = ds_equi7.sel(x=slice(4500000,5390000), y=slice(1790000,1200000)).expand_dims({"time": [sensing_time]})

**Visualizing**

The next steps will visualize the original, resampled and reprojected data

In [ ]:
transformer = Transformer.from_crs("EPSG:27704", "EPSG:4326", always_xy=True)
x,y = 5400000, 1800000
lon_max, lat_max = transformer.transform(x, y)
x,y = 4500000, 1200000
lon_min, lat_min = transformer.transform(x,y)

In [ ]:
X, Y = np.meshgrid(ds_aut["x"].values, ds_aut["y"].values)
lons_new, lats_new = transformer.transform(X,Y)

In [ ]:
X, Y = np.meshgrid(ds_equi7["x"].values, ds_equi7["y"].values)
lons_new_equi, lats_new_equi = transformer.transform(X,Y)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(17, 8), subplot_kw={"projection": ccrs.PlateCarree()})

axs=axs.flatten()
axs[0].pcolormesh(lon_grid_full, lat_grid_full, gridded_data["carbonmonoxide_total_column"], cmap="viridis")#, vmin=-2, vmax=1)
axs[3].pcolormesh(lons_new, lats_new, ds_aut["carbonmonoxide_total_column"].squeeze(), transform=ccrs.PlateCarree(), cmap="viridis")#, vmin=-2, vmax=1)
axs[2].pcolormesh(lon_original, lat_original, ds_band["carbonmonoxide_total_column"], cmap="viridis")#, vmin=-2, vmax=1)
axs[1].pcolormesh(lons_new_equi, lats_new_equi, ds_equi7["carbonmonoxide_total_column"], cmap="viridis")#, vmin=-2, vmax=1)

for ax in axs:
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    ax.gridlines(draw_labels=True, color="gray", linestyle="--")
    ax.set_xlim(lon_min-1, lon_max+1)
    ax.set_ylim(lat_min-1, lat_max+1)

axs[0].set_title("Regridded Sentinel-5")
axs[1].set_title("Reprojected Sentinel-5")
axs[2].set_title("Original Sentinel-5")
axs[3].set_title("Cropped reprojected")

plt.tight_layout()
plt.show()